# ThreatIntel-Agent Colab Quickstart

이 노트북은 Google Colab에서 프로젝트를 설치하고 FastAPI 서버를 실행하는 절차를 정리합니다.


In [ ]:
# 1. 프로젝트 업로드 또는 Git clone 후 경로로 이동
# %cd /content/threat-analysis-agent

# 2. 의존성 설치
!pip install -r requirements.txt


In [ ]:
# 3. 환경변수 설정 예시
import os

os.environ['DB_HOST'] = '<NEON_HOST>'
os.environ['DB_PORT'] = '5432'
os.environ['DB_NAME'] = '<DB_NAME>'
os.environ['DB_USER'] = '<DB_USER>'
os.environ['DB_PASSWORD'] = '<DB_PASSWORD>'
os.environ['LLM_BASE_URL'] = 'https://api.openai.com/v1'
os.environ['LLM_API_KEY'] = '<OPENAI_API_KEY>'
os.environ['LLM_MODEL'] = 'gpt-4o-mini'
os.environ['USE_VANNA'] = 'true'
os.environ['USE_LANGGRAPH'] = 'true'
os.environ['LANGSMITH_TRACING'] = 'false'
os.environ['LANGSMITH_TRACING_V2'] = 'false'
os.environ['LANGCHAIN_TRACING_V2'] = 'false'
os.environ['LANGSMITH_API_KEY'] = '<LANGSMITH_API_KEY_OPTIONAL>'
os.environ['LANGSMITH_PROJECT'] = 'threat-analysis-agent'


In [ ]:
# 4. 선택: CISA KEV CSV 적재
# CSV를 Colab 런타임 또는 Drive에 업로드한 뒤 경로를 지정하세요.
# !python scripts/import_cisa_kev.py /content/CISA_Known_Exploited_Vulnerabilities.csv


In [ ]:
# 5. 대시보드 API smoke test
from web_app import dashboard
dashboard()


In [ ]:
# 6. LangGraph Agentic Trace smoke test
from agent import ThreatIntelAgent

agent = ThreatIntelAgent()
events = []
for event in agent.run_stream('CISA KEV 카탈로그에서 최근 추가된 취약점 5개를 보여줘'):
    events.append(event)
    if event.get('type') == 'step':
        print(event.get('step'), event.get('status'), event.get('detail'))
    if event.get('type') == 'final':
        print('final rows:', len(event['result'].get('rows') or []))


In [ ]:
# 7. FastAPI 실행
# Colab 외부 접속은 ngrok/cloudflared 같은 터널 도구를 별도로 사용하세요.
!uvicorn web_app:app --host 0.0.0.0 --port 8080


In [ ]:
# 8. 선택: Ragas SQL 평가 실행
!python eval/ragas_eval.py
# LLM 기반 SQL semantic equivalence까지 실행하려면:
# !python eval/ragas_eval.py --ragas
